# LendingClub Credit Portfolio - business analysis
SQL controls, descriptive analysis, reproducible findings and recommendations. No predictive model.

In [1]:
from pathlib import Path
import sqlite3, json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == '04_Python' else Path.cwd()
con = sqlite3.connect(ROOT / '03_SQL/analysis.sqlite')

In [2]:
sql = (ROOT / '03_SQL/business_analysis.sql').read_text()
import re
for name, query in re.findall(r'-- name: (\w+)\n(.*?);', sql, re.S):
    table = pd.read_sql_query(query, con)
    print('\n'+name+'\n'+table.head(12).to_string(index=False))


portfolio_kpis
 loans  funded_amount  loan_amount  interest_rate  resolved_loans  defaults  default_rate  fully_paid_rate  current_share  interest_received  payments_received      dti  net_return_proxy
 39717      434810325 11219.443815       0.120212           38577    5627.0      0.145864         0.854136       0.028703        89905910.21       4.827044e+08 13.31513          0.099233

grade_risk_return
grade  loans  resolved_loans  funded_amount  default_rate  coupon_rate  interest_received  recovery_rate  net_return_proxy
    A  10085           10045       84738425      0.059930     0.073353         8501028.82       0.080463          0.071558
    B  12020           11675      130553250      0.122056     0.110212        22326216.78       0.076307          0.095227
    C   8098            7834       87290475      0.171943     0.135530        19162644.00       0.087703          0.104200
    D   5307            5085       64054375      0.219862     0.157199        16665553.83       0.0

In [3]:
findings = json.loads((ROOT / '08_Documentation/findings.json').read_text())
print('FINDINGS\n'+'\n'.join(findings['findings']))
print('\nRECOMMENDATIONS\n'+'\n'.join(findings['recommendations']))
con.close()

FINDINGS
39,717 loans in the 2007-2011 release funded USD 434,810,325.
38,577 resolved loans had 5,627 charged-off outcomes (14.59%).
1,140 current loans (2.87%) were excluded from resolved-outcome default and return rates.
Grade G had the highest observed resolved-loan net cash return proxy (13.63%); this is cumulative and not annualized.
Resolved net cash return proxy was 9.92% before servicing fees, tax and time value.
Origination FICO is absent in this release. FICO analysis and average FICO are explicitly unavailable.

RECOMMENDATIONS
Use the grade risk-return table to design diversification limits, then calculate dated cash-flow IRR and fees before making allocation decisions.
Investigate high-default purpose and DTI segments only after controlling for grade, term and vintage.
Compare vintages within matched loan terms and maturity windows; snapshot outcomes alone cannot establish underwriting improvement.
Add a release containing origination FICO and loan payment schedules to co

In [4]:
correlations = pd.read_csv(ROOT / '08_Documentation/spearman_correlations.csv', index_col=0)
print(correlations.to_string())
# Correlations are descriptive and cannot establish causation.

            loan_amnt  int_rate       dti  annual_inc  revol_util
loan_amnt    1.000000  0.254274  0.074378    0.428550    0.070672
int_rate     0.254274  1.000000  0.118745    0.060508    0.470814
dti          0.074378  0.118745  1.000000   -0.102938    0.272405
annual_inc   0.428550  0.060508 -0.102938    1.000000    0.047715
revol_util   0.070672  0.470814  0.272405    0.047715    1.000000


## Interpretation limits
- Selected older accepted-loan release covers 2007-2011, not the full 2018Q4 release. Outcomes are a historical snapshot, not current borrower status.
- Origination FICO fields are absent. FICO bands and average FICO are unavailable, not imputed.
- Current loans are unresolved. Default and return denominators include only Fully Paid and Charged Off/Default.
- Net cash return proxy is cumulative and funding-weighted, not annualized IRR or a volatility-adjusted ratio. Servicing fees, tax and dated cash-flow timing are unavailable.
- total_pymnt includes recoveries. Adding recoveries again would double-count cash. Collection recovery fees are subtracted.
- Grade, term and maturity differences confound vintage comparisons. No causal underwriting deterioration or investment recommendation is claimed.